# Regressleuth quickstart

A model's accuracy dropped after a deploy that changed several things at once. Which change, or which *pair* of changes, caused it?

Run this notebook top to bottom. No data or GPU needed.


In [ ]:
%pip install -q regressleuth  # or: %pip install -q -e ..  from a clone

## 1. Describe what changed and how to measure it

`run(components)` receives a dict where each component holds its **old or new** version, and returns the metric on your *fixed* evaluation cases. Here it is a stand-in function with a hidden interaction: the new model only breaks when paired with the new preprocessing.

In [ ]:
from regressleuth import diagnose

good = {"model": "v4", "preprocessing": "a1b2c3", "config": "cfg_old", "data": "2026-09", "threshold": 0.5}
bad = {"model": "v5", "preprocessing": "d4e5f6", "config": "cfg_new", "data": "2026-10", "threshold": 0.45}


def run(components):
    broken = components["model"] == "v5" and components["preprocessing"] == "d4e5f6"
    return 0.71 if broken else 0.94  # replace with: build the pipeline, score the fixed cases

In [ ]:
report = diagnose(good, bad, run, threshold=0.05, budget=16)
report

The answer takes 13 replays (9 to isolate the cause, 4 to audit it) instead of the 32 that trying every combination of 5 changes would need.


## 2. See every combination it tried


In [ ]:
from IPython.display import HTML

HTML(report.to_html())

## 3. Noisy metrics

If the metric varies run to run (training seeds, sampling), accept a `seed` argument and ask for repeats; the verdict then uses a confidence interval instead of a single number.


In [ ]:
import random


def noisy_run(components, seed):
    rng = random.Random(f"{sorted(components.items())}-{seed}")
    return run(components) + rng.gauss(0, 0.015)


diagnose(good, bad, noisy_run, threshold=0.05, repeats=4, max_repeats=16)

## 4. What the statuses mean

`verified`: reproduced, minimal, audit passed. `inferred`: leading suspect, some checks could not run. `assumption_violated`: the failure is not a simple one- or two-component effect, so no cause is claimed. See the [tutorial](../../docs/TUTORIAL.md).
